# 03 — Data Quality Checks

**Input:** cleaned files in `data/cleaned/`
**Output:** a validation report — pass/fail per check, plus written findings

## Why this notebook exists

After cleaning, we need to *formally validate* the result. Cleaning fixes
known issues; quality checks confirm the fixes worked and surface any
remaining problems before analysis begins.

## Quality dimensions checked

1. **Completeness** — no unexpected missing values
2. **Uniqueness** — primary keys are unique, no duplicate rows
3. **Validity** — values fall in sensible ranges and known sets
4. **Consistency** — foreign keys resolve across tables
5. **Business rules** — prices ≥ costs, no negative stock, etc.

## Approach

Every check produces one of: **PASS**, **WARN**, or **FAIL**.
Results are collected into a summary table at the end.

In [1]:
import pandas as pd
from pathlib import Path

# Path to cleaned data
base = Path("C:/Users/selin/anaconda_projects/data science/Retail-Sales-Inventory-Project/data/cleaned")

# Load cleaned tables
# NOTE: parse_dates=["Date"] re-parses the date column, since CSV
# flattens datetimes back to strings on save.
sales     = pd.read_csv(base / "sales_clean.csv",     parse_dates=["Date"])
products  = pd.read_csv(base / "products_clean.csv")
stores    = pd.read_csv(base / "stores_clean.csv",    parse_dates=["Store_Open_Date"])
inventory = pd.read_csv(base / "inventory_clean.csv")

# Quick sanity: shapes + dtypes
for name, df in {"sales": sales, "products": products,
                 "stores": stores, "inventory": inventory}.items():
    print(f"{name:10s} {df.shape[0]:>7,} rows × {df.shape[1]} cols")
print()
print(sales.dtypes)
print(stores.dtypes)

sales      829,262 rows × 5 cols
products        35 rows × 5 cols
stores          50 rows × 5 cols
inventory    1,593 rows × 3 cols

Sale_ID                int64
Date          datetime64[ns]
Store_ID               int64
Product_ID             int64
Units                  int64
dtype: object
Store_ID                    int64
Store_Name                 object
Store_City                 object
Store_Location             object
Store_Open_Date    datetime64[ns]
dtype: object


## Results collector

Every check appends its outcome to a `results` list. At the end, we
turn it into a scorecard dataframe.

In [2]:
results = []   # list of dicts: {"dimension", "check", "result", "detail"}

def record(dimension, check, passed, detail="", warn=False):
    """Append one check result to the scorecard."""
    if passed:
        status = "PASS"
    elif warn:
        status = "WARN"
    else:
        status = "FAIL"
    results.append({
        "dimension": dimension,
        "check":     check,
        "result":    status,
        "detail":    detail,
    })
    print(f"[{status}] {dimension} — {check}  {detail}")

## 1. Completeness

Are there missing values where there shouldn't be?

We check:
- `sales` — every row should have Sale_ID, Date, Store_ID, Product_ID, Units
- `products` — every product needs a name, category, cost, price
- `stores` — every store needs a name, city, location, open date
- `inventory` — every row needs both keys and stock value

Even one missing value in a key column would be a **FAIL**.

In [3]:
# Check: no nulls in any table
tables = {"sales": sales, "products": products,
          "stores": stores, "inventory": inventory}

for name, df in tables.items():
    n_missing = df.isnull().sum().sum()
    record(
        dimension="Completeness",
        check=f"{name}: no missing values",
        passed=(n_missing == 0),
        detail=f"({n_missing} nulls found)"
    )

[PASS] Completeness — sales: no missing values  (0 nulls found)
[PASS] Completeness — products: no missing values  (0 nulls found)
[PASS] Completeness — stores: no missing values  (0 nulls found)
[PASS] Completeness — inventory: no missing values  (0 nulls found)


## 2. Uniqueness

Each table has a natural primary key. Duplicates would break joins
and inflate aggregates.

- `sales.Sale_ID` — must be unique
- `products.Product_ID` — must be unique
- `stores.Store_ID` — must be unique
- `inventory.(Store_ID, Product_ID)` — composite key, must be unique

In [4]:
# Check: primary keys are unique
# Single-column keys
for df, key, label in [
    (sales,    "Sale_ID",    "sales.Sale_ID"),
    (products, "Product_ID", "products.Product_ID"),
    (stores,   "Store_ID",   "stores.Store_ID"),
]:
    n_dups = df[key].duplicated().sum()
    record(
        dimension="Uniqueness",
        check=f"{label} is unique",
        passed=(n_dups == 0),
        detail=f"({n_dups} duplicates)"
    )

# Composite key for inventory
n_dups = inventory.duplicated(subset=["Store_ID", "Product_ID"]).sum()
record(
    dimension="Uniqueness",
    check="inventory.(Store_ID, Product_ID) is unique",
    passed=(n_dups == 0),
    detail=f"({n_dups} duplicates)"
)

# Full-row duplicates in each table
for name, df in tables.items():
    n_dups = df.duplicated().sum()
    record(
        dimension="Uniqueness",
        check=f"{name}: no full-row duplicates",
        passed=(n_dups == 0),
        detail=f"({n_dups} found)"
    )

[PASS] Uniqueness — sales.Sale_ID is unique  (0 duplicates)
[PASS] Uniqueness — products.Product_ID is unique  (0 duplicates)
[PASS] Uniqueness — stores.Store_ID is unique  (0 duplicates)
[PASS] Uniqueness — inventory.(Store_ID, Product_ID) is unique  (0 duplicates)
[PASS] Uniqueness — sales: no full-row duplicates  (0 found)
[PASS] Uniqueness — products: no full-row duplicates  (0 found)
[PASS] Uniqueness — stores: no full-row duplicates  (0 found)
[PASS] Uniqueness — inventory: no full-row duplicates  (0 found)


## 3. Validity — numeric ranges

Values should fall in sensible ranges. We check:

- `Units` — integers, ≥ 1 (a sale of 0 units makes no sense)
- `Product_Price` / `Product_Cost` — > 0
- `Stock_On_Hand` — ≥ 0
- `Sale_ID`, `Store_ID`, `Product_ID` — positive integers

In [5]:
# Check: numeric values in sensible ranges
checks = [
    # (df,   column,          test_fn,                label
    (sales,     "Units",          lambda s: s >= 1,   "sales.Units ≥ 1",
    (products,  "Product_Price",  lambda s: s > 0,    "products.Product_Price > 0",
    (products,  "Product_Cost",   lambda s: s > 0,    "products.Product_Cost > 0",
    (inventory, "Stock_On_Hand",  lambda s: s >= 0,   "inventory.Stock_On_Hand ≥ 0",
]

for df, col, test, label, warn_only in checks:
    n_bad = (~test(df[col])).sum()
    record(
        dimension="Validity",
        check=label,
        passed=(n_bad == 0),
        detail=f"({n_bad} violations)",
        warn=warn_only
    )

# Special case: zero stock is legitimate (out-of-stock) — WARN not FAIL
n_zero = (inventory["Stock_On_Hand"] == 0).sum()
record(
    dimension="Validity",
    check="inventory has zero-stock rows",
    passed=(n_zero == 0),
    detail=f"({n_zero} rows — out-of-stock is valid)",
    warn=True   # if this "fails", it's only a WARN
)

[PASS] Validity — sales.Units ≥ 1  (0 violations)
[PASS] Validity — products.Product_Price > 0  (0 violations)
[PASS] Validity — products.Product_Cost > 0  (0 violations)
[PASS] Validity — inventory.Stock_On_Hand ≥ 0  (0 violations)
[WARN] Validity — inventory has zero-stock rows  (77 rows — out-of-stock is valid)


## 4. Validity — date ranges

- `sales.Date` — should fall within the project window (2022–2023)
- `stores.Store_Open_Date` — should not be in the future
- Both columns should be datetime (not strings)

In [6]:
# Check: dates are valid and in expected ranges
# Dtypes are datetime
record("Validity", "sales.Date is datetime",
       passed=str(sales["Date"].dtype).startswith("datetime"),
       detail=f"({sales['Date'].dtype})")

record("Validity", "stores.Store_Open_Date is datetime",
       passed=str(stores["Store_Open_Date"].dtype).startswith("datetime"),
       detail=f"({stores['Store_Open_Date'].dtype})")

# Date range sanity (adjust the expected bounds to your project window)
expected_start = pd.Timestamp("2022-01-01")
expected_end   = pd.Timestamp("2023-12-31")

out_of_range = (
    (sales["Date"] < expected_start) |
    (sales["Date"] > expected_end)
).sum()
record("Validity", "sales.Date within expected window",
       passed=(out_of_range == 0),
       detail=f"({out_of_range} rows outside {expected_start.date()}–{expected_end.date()})")

# Store open dates should be in the past
future_dates = (stores["Store_Open_Date"] > pd.Timestamp.today()).sum()
record("Validity", "stores.Store_Open_Date is not in the future",
       passed=(future_dates == 0),
       detail=f"({future_dates} future dates)")

[PASS] Validity — sales.Date is datetime  (datetime64[ns])
[PASS] Validity — stores.Store_Open_Date is datetime  (datetime64[ns])
[PASS] Validity — sales.Date within expected window  (0 rows outside 2022-01-01–2023-12-31)
[PASS] Validity — stores.Store_Open_Date is not in the future  (0 future dates)


## 5. Validity — allowed categorical values

Categorical columns should only contain known values. Any unexpected
value is usually a typo or a new category that needs review.

- `Product_Category` — should be from a small known set
- `Store_Location` — should be Downtown / Mall / Airport / etc.

We can't hard-code the expected values (they depend on the dataset),
so the check is: "the number of unique values is small AND there are
no case/whitespace variants."

In [7]:
# Check: categorical columns are clean
for col, df, label in [
    ("Product_Category", products, "products.Product_Category"),
    ("Store_Location",   stores,   "stores.Store_Location"),
]:
    # A common red flag: values that differ only by case/whitespace
    raw_vals   = df[col].dropna().unique()
    norm_vals  = {v.strip().lower() for v in raw_vals}
    n_raw      = len(raw_vals)
    n_norm     = len(norm_vals)

    record("Validity", f"{label} has no case/space variants",
           passed=(n_raw == n_norm),
           detail=f"({n_raw} raw, {n_norm} normalised)")

[PASS] Validity — products.Product_Category has no case/space variants  (5 raw, 5 normalised)
[PASS] Validity — stores.Store_Location has no case/space variants  (4 raw, 4 normalised)


## 6. Consistency — referential integrity

Foreign keys should resolve. If `sales.Store_ID` contains an ID that
isn't in `stores`, the sale is an orphan — it can't be enriched with
store info.

Cross-table checks:

| Child | Parent | Key |
|-------|--------|-----|
| sales | stores | Store_ID |
| sales | products | Product_ID |
| inventory | stores | Store_ID |
| inventory | products | Product_ID |

In [8]:
# Check: foreign keys resolve
fk_checks = [
    ("sales → stores",     sales,     "Store_ID",   stores,    "Store_ID"),
    ("sales → products",   sales,     "Product_ID", products,  "Product_ID"),
    ("inventory → stores", inventory, "Store_ID",   stores,    "Store_ID"),
    ("inventory → products", inventory, "Product_ID", products, "Product_ID"),
]

for label, child, ckey, parent, pkey in fk_checks:
    orphans = ~child[ckey].isin(parent[pkey])
    n_orphans = orphans.sum()
    record("Consistency", f"{label} (FK resolves)",
           passed=(n_orphans == 0),
           detail=f"({n_orphans} orphan rows)")

[PASS] Consistency — sales → stores (FK resolves)  (0 orphan rows)
[PASS] Consistency — sales → products (FK resolves)  (0 orphan rows)
[PASS] Consistency — inventory → stores (FK resolves)  (0 orphan rows)
[PASS] Consistency — inventory → products (FK resolves)  (0 orphan rows)


## 7. Business rules

Domain-specific rules that only make sense in this project:

- Every product's selling price should be **≥** its cost (else it's a loss)
- Every sale should have a matching inventory record
  *(41 store/product pairs sell but are missing from the inventory snapshot —
  treat as WARN, not FAIL, because inventory is a point-in-time snapshot)*
- Every store in `stores` should appear in `sales` at least once

In [11]:
# Check: business rules
# 7.1 Price ≥ Cost for every product
loss_makers = (products["Product_Price"] < products["Product_Cost"]).sum()
record("Business rules", "Product_Price ≥ Product_Cost",
       passed=(loss_makers == 0),
       detail=f"({loss_makers} products sold at a loss)")

# 7.2 Sales rows have matching inventory records
sales_keys = sales[["Store_ID", "Product_ID"]].drop_duplicates()
inv_keys   = inventory[["Store_ID", "Product_ID"]]
merged = sales_keys.merge(inv_keys, on=["Store_ID", "Product_ID"],
                          how="left", indicator=True)
n_missing = (merged["_merge"] == "left_only").sum()
record("Business rules", "Sales have matching inventory records",
       passed=(n_missing == 0),
       detail=f"({n_missing} unmatched — expected ~6,232)",
       warn=True)   # we already know this happens; it's informational

# 7.3 Every store has at least one sale
stores_with_sales = sales["Store_ID"].nunique()
n_stores = stores["Store_ID"].nunique()
record("Business rules", "Every store appears in sales",
       passed=(stores_with_sales == n_stores),
       detail=f"({stores_with_sales}/{n_stores} stores have sales)")

# 7.4 Every product has at least one sale
products_with_sales = sales["Product_ID"].nunique()
n_products = products["Product_ID"].nunique()
record("Business rules", "Every product appears in sales",
       passed=(products_with_sales == n_products),
       detail=f"({products_with_sales}/{n_products} products sold)")

[PASS] Business rules — Product_Price ≥ Product_Cost  (0 products sold at a loss)
[WARN] Business rules — Sales have matching inventory records  (41 unmatched — expected ~6,232)
[PASS] Business rules — Every store appears in sales  (50/50 stores have sales)
[PASS] Business rules — Every product appears in sales  (35/35 products sold)


## 8. Scorecard

All checks collected into a single table for review.

In [10]:
scorecard = pd.DataFrame(results)

# Sort so FAILs bubble to the top, then WARNs, then PASSes
order = {"FAIL": 0, "WARN": 1, "PASS": 2}
scorecard["_order"] = scorecard["result"].map(order)
scorecard = scorecard.sort_values(["_order", "dimension"]).drop(columns="_order")
scorecard = scorecard.reset_index(drop=True)

# Summary counts
print("Summary")
print(scorecard["result"].value_counts())
print()

# Full scorecard
scorecard

Summary
result
PASS    29
WARN     2
Name: count, dtype: int64



,dimension,check,result,detail
0,Business rules,Sales have matching inventory records,WARN,"(41 unmatched — expected ~6,232)"
1,Validity,inventory has zero-stock rows,WARN,(77 rows — out-of-stock is valid)
2,Business rules,Product_Price ≥ Product_Cost,PASS,(0 products sold at a loss)
3,Business rules,Every store appears in sales,PASS,(50/50 stores have sales)
4,Business rules,Every product appears in sales,PASS,(35/35 products sold)
5,Completeness,sales: no missing values,PASS,(0 nulls found)
6,Completeness,products: no missing values,PASS,(0 nulls found)
7,Completeness,stores: no missing values,PASS,(0 nulls found)
8,Completeness,inventory: no missing values,PASS,(0 nulls found)
9,Consistency,sales → stores (FK resolves),PASS,(0 orphan rows)


## 9. Findings

**Overall verdict:** PASS / PASS with warnings / FAIL *(fill in after running)*

### Passed
- No missing values in any table
- All primary keys unique
- All foreign keys resolve
- All date columns are valid datetimes in the expected range
- No negative values in any numeric column

### Warnings
- 77 inventory rows with zero stock — treated as valid out-of-stock state
- ~6,232 sales rows have no matching inventory record
  — these represent store/product combinations that sold but aren't
    tracked in the current inventory snapshot

### Actions for the analysis stage
- Exclude or flag zero-stock rows when calculating inventory value
- When joining sales → inventory, keep those 6,232 rows with NaN stock
  so revenue totals remain complete
- (Add any new findings here as you discover them)

---

## Next notebook

`04_analysis.ipynb` — uses the validated cleaned data to answer:
- Which products and categories drive the most revenue?
- Which stores / cities / locations perform best?
- How has revenue trended over time?
- How much capital is tied up in inventory?